In [0]:
# Configuration
CATALOG = "workspace"

BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

DATA_PATH = "/Volumes/workspace/bronze/tseries"

print(BRONZE, SILVER, GOLD)
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))


In [0]:
# Create schemas if they do not already exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))


## 1. Bronze layer — raw ingestion

Bronze preserves the source data as faithfully as possible. We add ingestion metadata but do not perform analytical transformations.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

files = {
    "telemetry": "PdM_telemetry.csv",
    "errors": "PdM_errors.csv",
    "failures": "PdM_failures.csv",
    "maint": "PdM_maint.csv",
    "machines": "PdM_machines.csv"
}

for name, file in files.items():
    path = f"{DATA_PATH}/{file}"
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv(path)
          .withColumn("_ingested_at", F.current_timestamp()))
    table = f"{BRONZE}.pdm_{name}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    print(f"Created {table}")

display(spark.sql(f"SHOW TABLES IN {BRONZE}"))


In [0]:
# Bronze validation
for name in files:
    table = f"{BRONZE}.pdm_{name}"
    print(f"--- {table} ---")
    display(spark.sql(f"SELECT COUNT(*) AS rows FROM {table}"))
    display(spark.sql(f"SELECT * FROM {table} LIMIT 5"))


# 2. Silver layer — cleaning and standardization

Silver will:
- cast timestamps explicitly;
- normalize IDs and categorical fields;
- remove impossible/null records;
- enforce uniqueness where appropriate;
- add useful date/time columns;
- preserve the analytical grain of each source.


In [0]:
# Telemetry
telemetry = (spark.table(f"{BRONZE}.pdm_telemetry")
    .withColumn("datetime", F.to_timestamp("datetime"))
    .withColumn("machineID", F.col("machineID").cast("int"))
    .withColumn("volt", F.col("volt").cast("double"))
    .withColumn("rotate", F.col("rotate").cast("double"))
    .withColumn("pressure", F.col("pressure").cast("double"))
    .withColumn("vibration", F.col("vibration").cast("double"))
    .filter("datetime IS NOT NULL AND machineID IS NOT NULL")
    .dropDuplicates(["datetime", "machineID"])
    .withColumn("date", F.to_date("datetime"))
    .withColumn("hour", F.hour("datetime"))
)

# Errors
errors = (spark.table(f"{BRONZE}.pdm_errors")
    .withColumn("datetime", F.to_timestamp("datetime"))
    .withColumn("machineID", F.col("machineID").cast("int"))
    .withColumn("errorID", F.trim(F.col("errorID")))
    .filter("datetime IS NOT NULL AND machineID IS NOT NULL AND errorID IS NOT NULL")
    .dropDuplicates(["datetime", "machineID", "errorID"])
)

# Failures
failures = (spark.table(f"{BRONZE}.pdm_failures")
    .withColumn("datetime", F.to_timestamp("datetime"))
    .withColumn("machineID", F.col("machineID").cast("int"))
    .withColumn("failure", F.trim(F.col("failure")))
    .filter("datetime IS NOT NULL AND machineID IS NOT NULL AND failure IS NOT NULL")
    .dropDuplicates(["datetime", "machineID", "failure"])
)

# Maintenance
maint = (spark.table(f"{BRONZE}.pdm_maint")
    .withColumn("datetime", F.to_timestamp("datetime"))
    .withColumn("machineID", F.col("machineID").cast("int"))
    .withColumn("comp", F.trim(F.col("comp")))
    .filter("datetime IS NOT NULL AND machineID IS NOT NULL AND comp IS NOT NULL")
    .dropDuplicates(["datetime", "machineID", "comp"])
)

# Machines
machines = (spark.table(f"{BRONZE}.pdm_machines")
    .withColumn("machineID", F.col("machineID").cast("int"))
    .withColumn("model", F.trim(F.col("model")))
    .withColumn("age", F.col("age").cast("double"))
    .dropDuplicates(["machineID"])
)

# Write Silver
for df, table in [
    (telemetry, f"{SILVER}.telemetry"),
    (errors, f"{SILVER}.errors"),
    (failures, f"{SILVER}.failures"),
    (maint, f"{SILVER}.maint"),
    (machines, f"{SILVER}.machines")
]:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)

print("Silver layer created.")


In [0]:
# Silver quality checks
quality_queries = {
    "telemetry_nulls": f'''
        SELECT
          SUM(CASE WHEN datetime IS NULL THEN 1 ELSE 0 END) AS null_datetime,
          SUM(CASE WHEN machineID IS NULL THEN 1 ELSE 0 END) AS null_machine,
          SUM(CASE WHEN volt IS NULL OR rotate IS NULL OR pressure IS NULL OR vibration IS NULL THEN 1 ELSE 0 END) AS null_sensor_rows
        FROM {SILVER}.telemetry
    ''',
    "failures": f"SELECT COUNT(*) AS rows, COUNT(DISTINCT machineID) AS machines, COUNT(DISTINCT failure) AS components FROM {SILVER}.failures",
    "errors": f"SELECT COUNT(*) AS rows, COUNT(DISTINCT errorID) AS error_types FROM {SILVER}.errors",
    "maintenance": f"SELECT COUNT(*) AS rows, COUNT(DISTINCT comp) AS components FROM {SILVER}.maint"
}
for name, q in quality_queries.items():
    print(name)
    display(spark.sql(q))


# Question 1 — Do non-breaking errors correlate with catastrophic failures?

### Method
For every failure:
- look backward 48 hours;
- count error events in the preceding 24h and 24–48h windows;
- compare error occurrence rates before failures;
- calculate the leading indicator strength for each error type.

A strong leading indicator should occur substantially more often immediately before failures than during a normal baseline period.


In [0]:
# Q1: errors in 0-24h and 24-48h before every failure
fail = spark.table(f"{SILVER}.failures").alias("f")
err = spark.table(f"{SILVER}.errors").alias("e")

error_before_failure = (
    fail.join(
        err,
        (F.col("f.machineID") == F.col("e.machineID")) &
        (F.col("e.datetime") > F.col("f.datetime") - F.expr("INTERVAL 48 HOURS")) &
        (F.col("e.datetime") <= F.col("f.datetime")),
        "left"
    )
    .select(
        F.col("f.machineID"),
        F.col("f.datetime").alias("failure_time"),
        F.col("f.failure"),
        F.col("e.errorID"),
        F.when(
            F.col("e.datetime") > F.col("f.datetime") - F.expr("INTERVAL 24 HOURS"), 1
        ).otherwise(0).alias("in_24h"),
        F.when(
            (F.col("e.datetime") <= F.col("f.datetime") - F.expr("INTERVAL 24 HOURS")) &
            (F.col("e.datetime") > F.col("f.datetime") - F.expr("INTERVAL 48 HOURS")), 1
        ).otherwise(0).alias("in_24_48h")
    )
)

q1_summary = (
    error_before_failure
    .groupBy("errorID")
    .agg(
        F.sum("in_24h").alias("events_0_24h"),
        F.sum("in_24_48h").alias("events_24_48h"),
        F.countDistinct(F.when(F.col("in_24h") == 1, F.struct("machineID","failure_time"))).alias("failure_cases_0_24h"),
        F.countDistinct(F.when(F.col("in_24_48h") == 1, F.struct("machineID","failure_time"))).alias("failure_cases_24_48h")
    )
    .orderBy(F.desc("failure_cases_0_24h"))
)

display(q1_summary)


In [0]:
# Q1: overall coverage of failures by errors
q1_overall = spark.sql(f'''
WITH failure_flags AS (
  SELECT
    f.machineID,
    f.datetime AS failure_time,
    CASE WHEN EXISTS (
      SELECT 1 FROM {SILVER}.errors e
      WHERE e.machineID=f.machineID
        AND e.datetime > f.datetime - INTERVAL 24 HOURS
        AND e.datetime <= f.datetime
    ) THEN 1 ELSE 0 END AS has_error_24h,
    CASE WHEN EXISTS (
      SELECT 1 FROM {SILVER}.errors e
      WHERE e.machineID=f.machineID
        AND e.datetime > f.datetime - INTERVAL 48 HOURS
        AND e.datetime <= f.datetime
    ) THEN 1 ELSE 0 END AS has_error_48h
  FROM {SILVER}.failures f
)
SELECT
  COUNT(*) AS failures,
  ROUND(100*AVG(has_error_24h),2) AS pct_with_error_24h,
  ROUND(100*AVG(has_error_48h),2) AS pct_with_error_48h
FROM failure_flags
''')
display(q1_overall)

# Gold table
q1_summary.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{GOLD}.q1_error_failure_leading_indicators")


### Q1 interpretation

The notebook output provides the evidence for whether errors are leading indicators. Pay particular attention to:
- `pct_with_error_24h`
- `pct_with_error_48h`
- the error type with the highest `failure_cases_0_24h`.

Do **not** claim that an error causes failure merely because it precedes failure; this is correlation/leading-indicator analysis.


# Question 2 — Machine age/model vs baseline telemetry

### Method
Join telemetry to machine metadata and calculate mean and standard deviation of each sensor by model and age band.

This separates:
- **level:** average baseline sensor value;
- **stability:** standard deviation;
- **age effect:** relationship between age and sensor baseline.


In [0]:
q2 = (
    spark.table(f"{SILVER}.telemetry").alias("t")
    .join(spark.table(f"{SILVER}.machines").alias("m"), "machineID")
    .withColumn(
        "age_band",
        F.when(F.col("age") < 5, "<5")
         .when(F.col("age") < 10, "5-9")
         .when(F.col("age") < 15, "10-14")
         .when(F.col("age") < 20, "15-19")
         .otherwise("20+")
    )
)

q2_model = q2.groupBy("model").agg(
    F.first("age").alias("representative_age"),
    F.avg("volt").alias("mean_volt"),
    F.stddev("volt").alias("sd_volt"),
    F.avg("rotate").alias("mean_rotate"),
    F.stddev("rotate").alias("sd_rotate"),
    F.avg("pressure").alias("mean_pressure"),
    F.stddev("pressure").alias("sd_pressure"),
    F.avg("vibration").alias("mean_vibration"),
    F.stddev("vibration").alias("sd_vibration")
).orderBy("model")

q2_age = q2.groupBy("age_band").agg(
    F.avg("volt").alias("mean_volt"),
    F.avg("rotate").alias("mean_rotate"),
    F.avg("pressure").alias("mean_pressure"),
    F.avg("vibration").alias("mean_vibration"),
    F.stddev("vibration").alias("sd_vibration")
).orderBy("age_band")

display(q2_model)
display(q2_age)

q2_model.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{GOLD}.q2_model_baseline")
q2_age.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{GOLD}.q2_age_baseline")


# Question 3 — Does time since maintenance affect sensor stability?

### Method
For each telemetry row, find the most recent maintenance event for that machine. Then calculate:
`days_since_maintenance = telemetry_time - latest_maintenance_time`

We compare sensor standard deviation and mean across elapsed-maintenance bands.


In [0]:
from pyspark.sql.window import Window
# Spark window solution: latest maintenance before each telemetry record
t = spark.table(f"{SILVER}.telemetry")
m = spark.table(f"{SILVER}.maint")

w = Window.partitionBy("machineID").orderBy("datetime").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Build a combined event stream so latest maintenance datetime is carried forward.
maintenance_times = m.select("machineID", F.col("datetime").alias("event_time"), F.col("datetime").alias("maint_time"))
telemetry_events = t.select("machineID", F.col("datetime").alias("event_time"), F.lit(None).cast("timestamp").alias("maint_time"))

events = telemetry_events.unionByName(maintenance_times).orderBy("machineID","event_time")
events = events.withColumn("last_maint_time", F.last("maint_time", ignorenulls=True).over(w))

t3 = (
    events.filter(F.col("maint_time").isNull())
    .join(t, ["machineID"], "inner")
    .filter(F.col("event_time") == F.col("datetime"))
    .withColumn("days_since_maintenance",
                (F.col("event_time").cast("long") - F.col("last_maint_time").cast("long")) / F.lit(86400.0))
)

t3 = t3.withColumn(
    "maintenance_age_band",
    F.when(F.col("days_since_maintenance").isNull(), "No prior maintenance")
     .when(F.col("days_since_maintenance") < 7, "0-7d")
     .when(F.col("days_since_maintenance") < 30, "7-30d")
     .when(F.col("days_since_maintenance") < 60, "30-60d")
     .when(F.col("days_since_maintenance") < 90, "60-90d")
     .when(F.col("days_since_maintenance") < 180, "90-180d")
     .otherwise("180d+")
)

q3 = t3.groupBy("maintenance_age_band").agg(
    F.count("*").alias("rows"),
    F.avg("volt").alias("mean_volt"),
    F.stddev("volt").alias("sd_volt"),
    F.avg("rotate").alias("mean_rotate"),
    F.stddev("rotate").alias("sd_rotate"),
    F.avg("pressure").alias("mean_pressure"),
    F.stddev("pressure").alias("sd_pressure"),
    F.avg("vibration").alias("mean_vibration"),
    F.stddev("vibration").alias("sd_vibration")
).orderBy("maintenance_age_band")

display(q3)
q3.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{GOLD}.q3_maintenance_sensor_stability")


# Question 4 — Forecast telemetry 12 hours ahead

### LSTM approach
We forecast **vibration** 12 hourly steps ahead using:
- 24-hour input sequence;
- machine-specific standardized vibration;
- LSTM encoder;
- 12-output regression head.

The prediction is then compared with a training-derived safety threshold.

> This is intentionally implemented in Python/Keras after the Silver layer has prepared the time-series data. The resulting forecast is written back to Gold.


In [0]:
# Prepare a manageable LSTM training set from Silver
# Run on Databricks ML Runtime.
import pandas as pd
import numpy as np

tele_pd = (spark.table(f"{SILVER}.telemetry")
           .select("datetime","machineID","vibration")
           .orderBy("machineID","datetime")
           .toPandas())

tele_pd["datetime"] = pd.to_datetime(tele_pd["datetime"])
tele_pd = tele_pd.sort_values(["machineID","datetime"])

# Standardize per machine using the full observed history for stable scaling.
tele_pd["vib_mean"] = tele_pd.groupby("machineID")["vibration"].transform("mean")
tele_pd["vib_std"] = tele_pd.groupby("machineID")["vibration"].transform("std")
tele_pd["vib_z"] = (tele_pd["vibration"] - tele_pd["vib_mean"]) / tele_pd["vib_std"].replace(0,1)

LOOKBACK = 24
HORIZON = 12

X, y, meta = [], [], []
# To keep the notebook practical, use every 3rd starting point.
for machine_id, g in tele_pd.groupby("machineID"):
    arr = g["vib_z"].to_numpy(dtype=np.float32)
    times = g["datetime"].to_numpy()
    for i in range(LOOKBACK, len(arr) - HORIZON + 1, 3):
        X.append(arr[i-LOOKBACK:i])
        y.append(arr[i:i+HORIZON])
        meta.append((int(machine_id), times[i]))

X = np.array(X, dtype=np.float32)[..., None]
y = np.array(y, dtype=np.float32)
meta = pd.DataFrame(meta, columns=["machineID","forecast_start"])

# Temporal split
cutoff = pd.Timestamp("2015-10-01")
train_mask = meta["forecast_start"] < cutoff
test_mask = ~train_mask

print("X:", X.shape, "y:", y.shape)
print("Train:", train_mask.sum(), "Test:", test_mask.sum())


In [0]:
from tensorflow import keras
from tensorflow.keras import layers

model_lstm = keras.Sequential([
    layers.Input(shape=(LOOKBACK, 1)),
    layers.LSTM(64),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(HORIZON)
])

model_lstm.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = model_lstm.fit(
    X[train_mask.values], y[train_mask.values],
    validation_split=0.1,
    epochs=10,
    batch_size=256,
    verbose=1
)

pred = model_lstm.predict(X[test_mask.values], batch_size=512)
mae = np.mean(np.abs(pred - y[test_mask.values]))
rmse = np.sqrt(np.mean((pred - y[test_mask.values])**2))
print("12-hour multi-step MAE (standardized vibration):", mae)
print("12-hour multi-step RMSE (standardized vibration):", rmse)


In [0]:
# Forecast the latest available 24h for every machine
latest_sequences = []
latest_meta = []

for machine_id, g in tele_pd.groupby("machineID"):
    g = g.sort_values("datetime")
    seq = g["vib_z"].tail(LOOKBACK).to_numpy(dtype=np.float32)
    if len(seq) == LOOKBACK:
        latest_sequences.append(seq)
        latest_meta.append({
            "machineID": int(machine_id),
            "last_observed": g["datetime"].max(),
            "vib_mean": float(g["vib_mean"].iloc[-1]),
            "vib_std": float(g["vib_std"].iloc[-1])
        })

latest_X = np.array(latest_sequences, dtype=np.float32)[...,None]
latest_pred_z = model_lstm.predict(latest_X, batch_size=128)

forecast_rows=[]
for i, meta_row in enumerate(latest_meta):
    for h in range(HORIZON):
        forecast_rows.append({
            "machineID": meta_row["machineID"],
            "last_observed": meta_row["last_observed"],
            "forecast_hour": h+1,
            "forecast_vibration": float(latest_pred_z[i,h] * meta_row["vib_std"] + meta_row["vib_mean"])
        })

forecast_pd = pd.DataFrame(forecast_rows)

# Threshold: 99th percentile of observed vibration.
vib_threshold = float(tele_pd["vibration"].quantile(0.99))
forecast_pd["threshold"] = vib_threshold
forecast_pd["proactive_alert"] = forecast_pd["forecast_vibration"] > vib_threshold

display(forecast_pd.sort_values(["proactive_alert","forecast_vibration"], ascending=[False,False]).head(50))

spark.createDataFrame(forecast_pd).write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{GOLD}.q4_lstm_vibration_forecast")


# Question 5 — Multi-sensor anomalies without standard error codes

### Method
Use Isolation Forest on multi-sensor observations:
- voltage
- rotation
- pressure
- vibration

Then join anomaly timestamps to errors. An observation is especially interesting when:
`anomaly = 1` AND `no_error_code_in_previous_24h = 1`.

This identifies structural sensor anomalies that were not accompanied by a recorded error.


In [0]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

anom_pd = (spark.table(f"{SILVER}.telemetry")
           .select("datetime","machineID","volt","rotate","pressure","vibration")
           .toPandas()
           .sort_values(["machineID","datetime"]))

features = ["volt","rotate","pressure","vibration"]
scaler = StandardScaler()
Z = scaler.fit_transform(anom_pd[features])

iso = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42,
    n_jobs=-1
)
anom_pd["anomaly_score"] = iso.decision_function(Z)
anom_pd["is_anomaly"] = (iso.predict(Z) == -1).astype(int)

# Error timestamps in preceding 24 hours
err_pd = spark.table(f"{SILVER}.errors").toPandas()
err_pd["datetime"] = pd.to_datetime(err_pd["datetime"])

# Efficient hourly/day join: create machine+date/hour key and use rolling-ish lookup.
err_pd["error_time"] = err_pd["datetime"]
err_counts = err_pd.groupby("machineID")["error_time"].apply(list).to_dict()

def has_recent_error(row):
    ts = row["datetime"]
    vals = err_counts.get(int(row["machineID"]), [])
    # Lists are short enough for this dataset; restrict to nearby timestamps.
    return any((ts - pd.Timedelta(hours=24)) < x <= ts for x in vals)

anom_pd["has_error_24h"] = anom_pd.apply(has_recent_error, axis=1)
anom_pd["unreported_anomaly"] = ((anom_pd["is_anomaly"] == 1) & (~anom_pd["has_error_24h"])).astype(int)

print("Anomaly rate:", anom_pd["is_anomaly"].mean())
print("Unreported anomaly rate:", anom_pd["unreported_anomaly"].mean())

gold5 = spark.createDataFrame(anom_pd)
gold5.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{GOLD}.q5_multisensor_anomalies")

display(gold5.orderBy(F.desc("unreported_anomaly"), F.asc("anomaly_score")).limit(50))


# Question 6 — Probability of failure within the next 24 hours

### Prediction target
For every hourly telemetry observation:

`failure_next_24h = 1` if the same machine experiences any component failure in the following 24 hours.

### Features
Build a 7-day rolling window:
- mean
- standard deviation
- minimum
- maximum

for voltage, rotation, pressure and vibration.

A temporal train/test split is used to avoid future leakage.


In [0]:
# Q6 feature engineering with Spark windows
from pyspark.sql.window import Window

t = spark.table(f"{SILVER}.telemetry").alias("t")
f = spark.table(f"{SILVER}.failures").select(
    F.col("machineID").alias("f_machineID"),
    F.col("datetime").alias("failure_time")
).alias("f")

# Create failure label using a range join.
labeled = (
    t.join(
        f,
        (F.col("t.machineID") == F.col("f.f_machineID")) &
        (F.col("f.failure_time") > F.col("t.datetime")) &
        (F.col("f.failure_time") <= F.col("t.datetime") + F.expr("INTERVAL 24 HOURS")),
        "left"
    )
    .select("t.*", F.when(F.col("f.failure_time").isNotNull(), 1).otherwise(0).alias("failure_next_24h"))
    .groupBy("datetime","machineID","volt","rotate","pressure","vibration")
    .agg(F.max("failure_next_24h").alias("failure_next_24h"))
)

w7 = Window.partitionBy("machineID").orderBy(F.col("datetime").cast("long")).rangeBetween(-7*24*3600, 0)

feature_df = labeled
for c in ["volt","rotate","pressure","vibration"]:
    feature_df = (feature_df
        .withColumn(f"{c}_mean_7d", F.avg(c).over(w7))
        .withColumn(f"{c}_std_7d", F.stddev(c).over(w7))
        .withColumn(f"{c}_min_7d", F.min(c).over(w7))
        .withColumn(f"{c}_max_7d", F.max(c).over(w7))
    )

feature_cols = [c for c in feature_df.columns if c.endswith(("_mean_7d","_std_7d","_min_7d","_max_7d"))]

q6_pd = feature_df.select(
    "datetime","machineID","failure_next_24h",*feature_cols
).dropna().toPandas()

q6_pd["datetime"] = pd.to_datetime(q6_pd["datetime"])
print(q6_pd["failure_next_24h"].value_counts(normalize=True))
print(q6_pd.shape)


In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

q6_pd = q6_pd.sort_values("datetime")
split = pd.Timestamp("2015-10-01")

train = q6_pd[q6_pd["datetime"] < split]
test = q6_pd[q6_pd["datetime"] >= split]

X_train = train[feature_cols]
y_train = train["failure_next_24h"]
X_test = test[feature_cols]
y_test = test["failure_next_24h"]

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

test["failure_probability_24h"] = rf.predict_proba(X_test)[:,1]

print("ROC-AUC:", roc_auc_score(y_test, test["failure_probability_24h"]))
print("PR-AUC:", average_precision_score(y_test, test["failure_probability_24h"]))
print(classification_report(y_test, (test["failure_probability_24h"] >= 0.5).astype(int), zero_division=0))

# Feature importance
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

display(importance.head(20))

# Gold risk table: latest available score per machine
latest_scores = (test.sort_values("datetime")
                 .groupby("machineID", as_index=False)
                 .tail(1)
                 [["machineID","datetime","failure_probability_24h"]]
                 .sort_values("failure_probability_24h", ascending=False))

display(latest_scores.head(20))

spark.createDataFrame(latest_scores).write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{GOLD}.q6_machine_failure_risk")
spark.createDataFrame(importance).write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{GOLD}.q6_feature_importance")


# Question 7 — Which component is most likely to fail next?

### Formulation
Use each historical failure as a supervised training example.

For each failure timestamp:
- calculate the preceding 24-hour telemetry statistics;
- calculate error counts in the preceding 24 hours;
- use the failure component (`comp1`–`comp4`) as the multiclass target.

The model predicts **which component** fails next, conditional on a failure occurring.


In [0]:
# Q7: failure-event feature table
fail_pd = spark.table(f"{SILVER}.failures").toPandas()
tele_pd2 = spark.table(f"{SILVER}.telemetry").toPandas()
err_pd2 = spark.table(f"{SILVER}.errors").toPandas()

for df in [fail_pd, tele_pd2, err_pd2]:
    df["datetime"] = pd.to_datetime(df["datetime"])

tele_pd2 = tele_pd2.sort_values(["machineID","datetime"])
err_pd2 = err_pd2.sort_values(["machineID","datetime"])

rows=[]
for _, fr in fail_pd.iterrows():
    m = int(fr.machineID)
    ft = fr.datetime
    hist = tele_pd2[
        (tele_pd2.machineID == m) &
        (tele_pd2.datetime <= ft) &
        (tele_pd2.datetime > ft - pd.Timedelta(hours=24))
    ]
    if len(hist) < 12:
        continue
    er = err_pd2[
        (err_pd2.machineID == m) &
        (err_pd2.datetime <= ft) &
        (err_pd2.datetime > ft - pd.Timedelta(hours=24))
    ]
    row = {
        "machineID": m,
        "failure_time": ft,
        "target": fr.failure,
        "error_count": len(er)
    }
    for c in ["volt","rotate","pressure","vibration"]:
        row[f"{c}_mean"] = hist[c].mean()
        row[f"{c}_std"] = hist[c].std()
        row[f"{c}_min"] = hist[c].min()
        row[f"{c}_max"] = hist[c].max()
        row[f"{c}_last"] = hist[c].iloc[-1]
    rows.append(row)

q7_pd = pd.DataFrame(rows).sort_values("failure_time")
print(q7_pd["target"].value_counts())
display(q7_pd.head())


In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

q7_features = [c for c in q7_pd.columns if c not in ["machineID","failure_time","target"]]

split7 = pd.Timestamp("2015-10-01")
train7 = q7_pd[q7_pd["failure_time"] < split7]
test7 = q7_pd[q7_pd["failure_time"] >= split7]

le = LabelEncoder()
ytr = le.fit_transform(train7["target"])
yte = le.transform(test7["target"])

rf7 = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf7.fit(train7[q7_features], ytr)

test7 = test7.copy()
test7["predicted_component"] = le.inverse_transform(rf7.predict(test7[q7_features]))

print("Accuracy:", accuracy_score(yte, le.transform(test7["predicted_component"])))
print("Macro F1:", f1_score(yte, le.transform(test7["predicted_component"]), average="macro"))
print(classification_report(yte, le.transform(test7["predicted_component"]), target_names=le.classes_, zero_division=0))

# Gold predictions
spark.createDataFrame(test7).write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{GOLD}.q7_component_predictions")

# Latest component risk distribution among machines' most recent failure-event feature rows
display(test7[["machineID","failure_time","target","predicted_component"]].tail(30))


# Question 8 — Remaining Useful Life (RUL) after maintenance

### Definition used
For a maintenance event on `(machineID, component)`:
- find the next failure of the same component on the same machine;
- RUL = elapsed days between maintenance and that next failure.

If no such failure occurs before the dataset ends, the observation is right-censored and is not used as an exact supervised target.

### Model
Use a Random Forest regression model with post-maintenance telemetry features:
- sensor mean/std/min/max;
- elapsed days since maintenance;
- machine age;
- component.

This is a practical baseline. A survival-analysis model would be more statistically rigorous for censored observations.


In [0]:
# Q8 create maintenance -> next same-component failure targets
maint_pd = spark.table(f"{SILVER}.maint").toPandas()
fail_pd = spark.table(f"{SILVER}.failures").toPandas()
machines_pd = spark.table(f"{SILVER}.machines").toPandas()
tele_pd3 = spark.table(f"{SILVER}.telemetry").toPandas()

for df in [maint_pd, fail_pd, tele_pd3]:
    df["datetime"] = pd.to_datetime(df["datetime"])

fail_groups = {
    (int(mid), comp): g.sort_values("datetime")
    for (mid, comp), g in fail_pd.groupby(["machineID","failure"])
}

rul_rows=[]
for _, mr in maint_pd.iterrows():
    key=(int(mr.machineID), mr.comp)
    future = fail_groups.get(key)
    if future is None:
        continue
    ff = future[future.datetime > mr.datetime]
    if ff.empty:
        continue
    failure_time = ff.iloc[0].datetime
    hist = tele_pd3[
        (tele_pd3.machineID == int(mr.machineID)) &
        (tele_pd3.datetime > mr.datetime) &
        (tele_pd3.datetime <= min(failure_time, mr.datetime + pd.Timedelta(days=7)))
    ]
    if len(hist) < 12:
        continue
    machine_age = machines_pd.loc[machines_pd.machineID == int(mr.machineID), "age"]
    row={
        "machineID": int(mr.machineID),
        "maintenance_time": mr.datetime,
        "comp": mr.comp,
        "age": float(machine_age.iloc[0]) if len(machine_age) else np.nan,
        "rul_days": (failure_time-mr.datetime).total_seconds()/86400
    }
    for c in ["volt","rotate","pressure","vibration"]:
        row[f"{c}_mean_7d"] = hist[c].mean()
        row[f"{c}_std_7d"] = hist[c].std()
        row[f"{c}_min_7d"] = hist[c].min()
        row[f"{c}_max_7d"] = hist[c].max()
    rul_rows.append(row)

rul_pd=pd.DataFrame(rul_rows).dropna()
print("Supervised RUL rows:", len(rul_pd))
display(rul_pd.head())


In [0]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

rul_pd=rul_pd.sort_values("maintenance_time")
rul_features=[c for c in rul_pd.columns if c not in ["machineID","maintenance_time","rul_days","comp"]]

# One-hot encode component
rul_model_pd=pd.get_dummies(rul_pd, columns=["comp"], dtype=int)
rul_features=[c for c in rul_model_pd.columns if c not in ["machineID","maintenance_time","rul_days"]]

split8=pd.Timestamp("2015-10-01")
tr8=rul_model_pd[rul_model_pd.maintenance_time < split8]
te8=rul_model_pd[rul_model_pd.maintenance_time >= split8]

rf8=RandomForestRegressor(
    n_estimators=300,
    max_depth=14,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)
rf8.fit(tr8[rul_features], tr8["rul_days"])
te8=te8.copy()
te8["predicted_rul_days"]=rf8.predict(te8[rul_features])

print("RUL MAE (days):", mean_absolute_error(te8["rul_days"],te8["predicted_rul_days"]))
print("RUL RMSE (days):", mean_squared_error(te8["rul_days"],te8["predicted_rul_days"])**0.5)

display(te8[["machineID","maintenance_time","rul_days","predicted_rul_days"]].head(30))

spark.createDataFrame(te8).write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{GOLD}.q8_rul_predictions")


# Question 9 — Are preventive maintenance events too early or preventing failures?

### What the supplied data can measure
For every scheduled maintenance event, find the next failure of the same component on the same machine.

Metrics:
- number of maintenance events followed by a same-component failure;
- median/mean days from maintenance to failure;
- percentage with failure within 7, 30, 60, and 90 days.

A very short maintenance-to-failure interval indicates that the maintenance event did not provide a long failure-free interval. A long interval indicates a longer observed post-maintenance period.

**Important:** “too early” strictly requires the planned maintenance interval or component-life target. Those fields are not in the supplied data, so the notebook does not invent them.


In [0]:
# Q9 using Spark SQL
q9 = spark.sql(f'''
WITH next_failure AS (
  SELECT
    m.machineID,
    m.datetime AS maintenance_time,
    m.comp,
    MIN(f.datetime) AS next_failure_time
  FROM {SILVER}.maint m
  LEFT JOIN {SILVER}.failures f
    ON m.machineID=f.machineID
   AND m.comp=f.failure
   AND f.datetime > m.datetime
  GROUP BY m.machineID,m.datetime,m.comp
),
metrics AS (
  SELECT *,
    (unix_timestamp(next_failure_time)-unix_timestamp(maintenance_time))/86400.0 AS days_to_same_component_failure
  FROM next_failure
)
SELECT
  COUNT(*) AS maintenance_events,
  COUNT(next_failure_time) AS events_with_observed_same_component_failure,
  ROUND(100*COUNT(next_failure_time)/COUNT(*),2) AS pct_with_observed_failure,
  ROUND(AVG(days_to_same_component_failure),2) AS mean_days_to_failure,
  ROUND(percentile_approx(days_to_same_component_failure,0.5),2) AS median_days_to_failure,
  ROUND(100*AVG(CASE WHEN days_to_same_component_failure <= 7 THEN 1 ELSE 0 END),2) AS pct_failure_within_7d,
  ROUND(100*AVG(CASE WHEN days_to_same_component_failure <= 30 THEN 1 ELSE 0 END),2) AS pct_failure_within_30d,
  ROUND(100*AVG(CASE WHEN days_to_same_component_failure <= 60 THEN 1 ELSE 0 END),2) AS pct_failure_within_60d,
  ROUND(100*AVG(CASE WHEN days_to_same_component_failure <= 90 THEN 1 ELSE 0 END),2) AS pct_failure_within_90d
FROM metrics
''')

display(q9)
spark.sql(f'''
CREATE OR REPLACE TABLE {GOLD}.q9_maintenance_effectiveness AS
WITH next_failure AS (
  SELECT
    m.machineID, m.datetime AS maintenance_time, m.comp,
    MIN(f.datetime) AS next_failure_time
  FROM {SILVER}.maint m
  LEFT JOIN {SILVER}.failures f
    ON m.machineID=f.machineID AND m.comp=f.failure AND f.datetime > m.datetime
  GROUP BY m.machineID,m.datetime,m.comp
)
SELECT *,
  (unix_timestamp(next_failure_time)-unix_timestamp(maintenance_time))/86400.0 AS days_to_same_component_failure
FROM next_failure
''')


# Question 10 — MTBF by machine model and maintenance/component context

The supplied `PdM_maint.csv` contains only:
- datetime
- machineID
- comp

There is **no maintenance team, technician, repair type, or repair-vs-replacement field**.

Therefore:
- team-level MTBF cannot be computed from this dataset;
- repair-type MTBF cannot be computed from this dataset;
- we can still calculate MTBF by **machine model** and **failed component**, which is the supported analysis.

MTBF here is calculated as the time between consecutive failures of the same component on the same machine.


In [0]:
q10 = spark.sql(f'''
WITH failure_intervals AS (
  SELECT
    f.machineID,
    m.model,
    f.failure AS component,
    f.datetime,
    LAG(f.datetime) OVER (
      PARTITION BY f.machineID, f.failure
      ORDER BY f.datetime
    ) AS previous_failure
  FROM {SILVER}.failures f
  JOIN {SILVER}.machines m USING(machineID)
)
SELECT
  model,
  component,
  COUNT(*) AS intervals,
  ROUND(AVG((unix_timestamp(datetime)-unix_timestamp(previous_failure))/86400.0),2) AS mean_mtbf_days,
  ROUND(percentile_approx((unix_timestamp(datetime)-unix_timestamp(previous_failure))/86400.0,0.5),2) AS median_mtbf_days
FROM failure_intervals
WHERE previous_failure IS NOT NULL
GROUP BY model, component
ORDER BY model, component
''')

display(q10)

spark.sql(f'''
CREATE OR REPLACE TABLE {GOLD}.q10_mtbf_model_component AS
WITH failure_intervals AS (
  SELECT
    f.machineID, m.model, f.failure AS component, f.datetime,
    LAG(f.datetime) OVER (PARTITION BY f.machineID, f.failure ORDER BY f.datetime) AS previous_failure
  FROM {SILVER}.failures f
  JOIN {SILVER}.machines m USING(machineID)
)
SELECT
  model,
  component,
  COUNT(*) AS intervals,
  AVG((unix_timestamp(datetime)-unix_timestamp(previous_failure))/86400.0) AS mean_mtbf_days,
  percentile_approx((unix_timestamp(datetime)-unix_timestamp(previous_failure))/86400.0,0.5) AS median_mtbf_days
FROM failure_intervals
WHERE previous_failure IS NOT NULL
GROUP BY model, component
''')


# 3. Final Gold-layer answer dashboard

The following tables are the main deliverables for the assignment:

| Question | Gold output |
|---|---|
| Q1 | `workspace.gold.q1_error_failure_leading_indicators` |
| Q2 | `workspace.gold.q2_model_baseline`, `workspace.gold.q2_age_baseline` |
| Q3 | `workspace.gold.q3_maintenance_sensor_stability` |
| Q4 | `workspace.gold.q4_lstm_vibration_forecast` |
| Q5 | `workspace.gold.q5_multisensor_anomalies` |
| Q6 | `workspace.gold.q6_machine_failure_risk`, `workspace.gold.q6_feature_importance` |
| Q7 | `workspace.gold.q7_component_predictions` |
| Q8 | `workspace.gold.q8_rul_predictions` |
| Q9 | `workspace.gold.q9_maintenance_effectiveness` |
| Q10 | `workspace.gold.q10_mtbf_model_component` |

These tables are the **Gold layer**: business-ready outputs rather than raw data.


In [0]:
# Gold inventory
display(spark.sql(f"SHOW TABLES IN {GOLD}"))

# Quick view of the most important outputs
display(spark.table(f"{GOLD}.q1_error_failure_leading_indicators"))
display(spark.table(f"{GOLD}.q2_model_baseline"))
display(spark.table(f"{GOLD}.q3_maintenance_sensor_stability"))
display(spark.table(f"{GOLD}.q6_machine_failure_risk").limit(20))
display(spark.table(f"{GOLD}.q10_mtbf_model_component"))


# Final conclusions to include in the report

After running the notebook, write the final report using the actual values generated in the Gold tables.

### Q1
State the percentage of failures preceded by an error in 24h and 48h, and identify which error IDs are most frequent before failures. Treat this as a leading-indicator relationship, not proof of causation.

### Q2
Compare model/age-band baseline means and standard deviations. If older bands show higher vibration or lower pressure, that supports age-adjusted thresholds; otherwise state that the observed dataset does not show a strong monotonic age effect.

### Q3
Compare sensor standard deviation across maintenance-age bands. Increasing standard deviation with elapsed time supports degradation; a flat pattern does not.

### Q4
Report LSTM MAE/RMSE and the number of machines whose 12-hour forecast exceeds the selected 99th-percentile vibration threshold.

### Q5
Report anomaly rate and the number/rate of anomalies without an error code in the preceding 24h.

### Q6
Report ROC-AUC, PR-AUC and the latest machine-level failure probabilities. Because failures are rare, PR-AUC is especially informative.

### Q7
Report multiclass accuracy and macro-F1, plus the confusion matrix/classification report. This predicts the component conditional on a failure event.

### Q8
Report RUL MAE/RMSE in days and show example predicted RUL values. Note that exact RUL labels are censored when no subsequent failure is observed.

### Q9
Report the percentage of maintenance events followed by a same-component failure and the failure-free interval distribution. Do not call maintenance “too early” unless a planned interval/target exists.

### Q10
Report MTBF by model/component. Explicitly state that team/repair-type comparisons are not possible because those columns are absent from the provided files.


## Recommended Databricks architecture diagram

```text
                  CSV FILES
                     |
                     v
        +---------------------------+
        |       BRONZE LAYER        |
        | raw Delta tables          |
        | telemetry / errors /      |
        | failures / maint /        |
        | machines                   |
        +-------------+-------------+
                      |
                      v
        +---------------------------+
        |        SILVER LAYER       |
        | typed + cleaned +         |
        | deduplicated + time-ready |
        | telemetry / errors /      |
        | failures / maint /        |
        | machines                  |
        +-------------+-------------+
                      |
          +-----------+-----------+
          |           |           |
          v           v           v
      EDA / SQL    ML Features   Time Series
          |           |           |
          +-----------+-----------+
                      |
                      v
        +---------------------------+
        |         GOLD LAYER        |
        | Q1-Q10 analytical outputs |
        | risk scores / forecasts   |
        | anomalies / RUL / MTBF    |
        +---------------------------+
                      |
                      v
              Databricks SQL /
              Dashboards / Report
```
